# Ethical & Sustainable Generative AI: A Framework for Responsible AI Decision-Making

**Author:** Bugatha Ganasyam Mani Sankar
**Institution:** JNTU-GV College of Engineering, Vizianagaram
**Mentor:** Mohammad Sameer (Phinerd)

This notebook implements the complete research pipeline for evaluating ethical and environmental impact of Generative AI systems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from ethical_analysis.bias_detector import EthicalRiskModel, BiasDetector, ExplainabilityScorer, RiskClassifier
from environmental_metrics.calculator import EnvironmentalCalculator, OptimizationRecommender
from models.rai_scorer import RAIScorer, ScenarioSimulator

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
print('✅ All modules imported successfully!')

## 1. Ethical Risk Modeling

In [ ]:
# Initialize ethical risk model
ethical_model = EthicalRiskModel(alpha1=0.4, alpha2=0.35, alpha3=0.25)

# Simulate data for 150 scenarios (50 per category)
np.random.seed(42)
n_scenarios = 150

scenarios = []
scenario_types = ['military'] * 50 + ['crisis'] * 50 + ['policy'] * 50

for i in range(n_scenarios):
    # Generate synthetic ground truth and predictions
    y_true = np.random.randint(0, 2, 100)
    y_pred = np.random.randint(0, 2, 100)
    protected_attr = np.random.choice(['male', 'female', 'other'], 100)
    
    # Scenario-specific bias injection
    if scenario_types[i] == 'military':
        bias_injection = 0.22
    elif scenario_types[i] == 'crisis':
        bias_injection = 0.15
    else:
        bias_injection = 0.18
    
    output_text = f"AI recommendation for {scenario_types[i]} scenario {i}"
    
    result = ethical_model.evaluate(
        y_true=y_true,
        y_pred=y_pred,
        protected_attr=protected_attr,
        scenario_type=scenario_types[i],
        output_text=output_text,
        attention_weights=np.random.dirichlet(np.ones(10)),
        lime_fidelity=np.random.uniform(0.7, 0.95)
    )
    
    result['scenario_type'] = scenario_types[i]
    scenarios.append(result)

ethical_df = pd.DataFrame(scenarios)
print('Ethical Evaluation Results:')
print(ethical_df.groupby('scenario_type')[['ethical_score', 'bias_score', 'explainability_score', 'risk_factor']].mean())

## 2. Environmental Impact Estimation

In [ ]:
# Initialize environmental calculator
env_calc = EnvironmentalCalculator(pue=1.4, carbon_intensity=450, wue=1.0)

# Calculate footprints for different models and tasks
models = ['GPT-4', 'Claude-3', 'Llama-3']
tasks = ['text', 'image', 'video']

env_results = []
for model in models:
    for task in tasks:
        if task == 'text':
            inference_time = 0.5
            num_gpus = 1
        elif task == 'image':
            inference_time = 2.0
            num_gpus = 2
        else:
            inference_time = 5.0
            num_gpus = 4
        
        footprint = env_calc.calculate_full_footprint(
            gpu_type='A100',
            num_gpus=num_gpus,
            inference_time_seconds=inference_time,
            batch_size=1,
            scenario='inference'
        )
        
        footprint['model'] = model
        footprint['task'] = task
        env_results.append(footprint)

env_df = pd.DataFrame(env_results)
print('Environmental Footprints:')
print(env_df.pivot(index='model', columns='task', values='energy_kwh'))

## 3. Responsible AI Scoring

In [ ]:
# Initialize RAI Scorer
rai_scorer = RAIScorer(w1=0.4, w2=0.35, w3=0.25)
simulator = ScenarioSimulator(rai_scorer)

# Simulate all scenarios
rai_results = []
for scenario_type in ['military', 'crisis', 'policy', 'medical', 'financial']:
    for model in models:
        params = {
            'ethical_score': np.random.uniform(55, 85),
            'energy_kwh': np.random.uniform(0.05, 0.2),
            'water_liters': np.random.uniform(0.1, 0.4),
            'carbon_kgco2e': np.random.uniform(0.02, 0.09),
            'data_disclosure': np.random.randint(20, 40),
            'architecture_transparency': np.random.randint(15, 30),
            'environmental_reporting': np.random.randint(10, 30)
        }
        
        result = simulator.simulate_scenario(scenario_type, params)
        result['model'] = model
        rai_results.append(result)

rai_df = pd.DataFrame(rai_results)
print('RAI Scores by Scenario and Model:')
print(rai_df.pivot(index='scenario_type', columns='model', values='rai_score'))

## 4. Optimization Recommendations

In [ ]:
# Generate optimization recommendations
optimizer = OptimizationRecommender()
current_footprint = {
    'energy_kwh': 0.18,
    'water_liters': 0.36,
    'carbon_kgco2e': 0.081
}

recommendations = optimizer.recommend(current_footprint, priority='balanced')
rec_df = pd.DataFrame(recommendations)
print('Optimization Recommendations:')
print(rec_df[['technique', 'energy_reduction_%', 'accuracy_loss_%']])

## 5. Visualization & Analysis

In [ ]:
# Generate all visualizations
from notebooks.visualization import *
generate_all_visualizations(ethical_df, env_df, rai_df, rec_df)
print('✅ All visualizations saved to results/ directory')